<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2B: *Reservoirs Sampling Grid Intersect*


## **Metadata**
---
### Contents  
> 1. *Geometries*
> 2. *Spatial Join of reservoirs with main dataset*
> 3. *Export File*
---
### Notes
---
### Inputs
- Main Dataset - `samples_weather_NDVI.csv` 
- Cleaned Reservoir Data - `reservoirs.csv` (cleaned in module 1)
---
### Outputs  
- `samples_res.csv` Main dataset merged with reservoir data
---
### User Created Dependencies  

In [ ]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

## Load Data

In [ ]:
samples = pd.read_csv("../data/processed/samples_weather_NDVI.csv")

# Raw Reservoir data is processed and cleaned in Appendix H.
reservoirs = pd.read_csv("../data/raw/reservoirs.csv")

## Geometries

In [ ]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep]

join_samples['geometry'] = [Point(xy) for xy in zip(join_samples ['centroid_easting'], join_samples ['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

reservoirs['geometry'] = [Point(xy) for xy in zip(reservoirs['Longitude'], reservoirs['Latitude'])]
reservoir_gdf = gpd.GeoDataFrame(reservoirs, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection to match grid CRS
reservoir_gdf_projected = reservoir_gdf.to_crs(3310)

## Reservoir Sampling Grid Intersect

**Main loop of Instersect algorithm**
- Loop through all dates in reservoir dataset
- Save all reservoir measurements associated with the current date
    - If no reservoir readings, move to next date
- Savle all grid centroids associated with the current date
- Create a buffer around each around each reservoir for the current date
- Intersection spatial join of sampling centroids and buffered reserovirs
- Aggregate reservoir info
    - `Total_reservoir_level` = sum of all reservoirs in a sampling area
    - `Total_reservoir_count` = count of all reservoirs in a sampling area
    - `Reservoir_has_data` = Indicates whether or not a measurement exists on the date
- Assign samples to new columns in temporary dataframe

In [ ]:
# Square Buffer radius, creates a 46000 * 46000 meter sized box
buffer_size = 23000

In [8]:
for dt in reservoir_gdf_projected['Date'].unique():
    
    # Fires on this date
    res_today = reservoir_gdf_projected[reservoir_gdf_projected['Date'] == dt]
    if res_today.empty:
        continue

    # Samples on this date
    samples_today = samples_gdf[samples_gdf['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each sample
    res_buffered = res_today.copy()
    res_buffered['geometry'] = res_buffered['geometry'].apply(lambda p: square_buffer(p, buffer_size))

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, res_buffered, how='left', predicate='intersects')
    
    joined['level_missing'] = 1 - joined['Reservoir Level_has_data']
    
    # Aggregate counts and total damage per sample
    grouped = joined.groupby('grid_id').agg(
        reservoir_count=('Station_ID', 'count'),
        total_reservoir_level=('Reservoir Level', 'sum'),
        stations_missing_levels = ('level_missing', 'sum')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_gdf.loc[samples_gdf['Date'] == dt, 'reservoir_count'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['reservoir_count']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'total_reservoir_level'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['total_reservoir_level']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'stations_missing_levels'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['stations_missing_levels']).fillna(0)

## Post Merge Checks

In [9]:
post_merge_check(samples_gdf, join_samples)

Premerged shape:  (608880, 5)
Merged shape:  (608880, 8)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


In [10]:
samples_gdf.isna().sum()

centroid_easting           0
centroid_northing          0
Date                       0
grid_id                    0
geometry                   0
reservoir_count            0
total_reservoir_level      0
stations_missing_levels    0
dtype: int64

**Merge temporary dataframe with main dataset**

In [ ]:
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])

# Merge on BOTH station and date
samples_res = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [13]:
post_merge_check(samples_res, samples)

Premerged shape:  (608880, 65)
Merged shape:  (608880, 68)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0


NA values after merge:  0


In [14]:
samples_res.isna().sum()

Sample_ID                         0
Date                              0
Burning Index                     0
Energy Release Component          0
Actual Evapotranspiration         0
                                 ..
NDVI_mean_difference              0
NDVI_mean_difference_has_value    0
reservoir_count                   0
total_reservoir_level             0
stations_missing_levels           0
Length: 68, dtype: int64

## Export File

In [15]:
samples_res.to_csv('../data/processed/samples_res.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
